In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier

In [2]:
df = pd.read_csv("titanic_train.csv")
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [3]:
print(df["Age"].isna().sum())
print(df["Age"].median())

177
28.0


In [5]:
age_median = df["Age"].median()
df["Sex"] = df["Sex"].map({"male": 0, "female": 1})
df["Age"] = df["Age"].fillna(age_median)
print(df[["Pclass", "Sex", "Age", "Fare"]].isna().sum())

Pclass    0
Sex       0
Age       0
Fare      0
dtype: int64


In [6]:
features = ["Pclass", "Sex", "Age", "Fare"]
x = df[features]
y = df["Survived"]
x_train, x_test, y_train, y_test = train_test_split(
    x, y, test_size=0.3, random_state=42)
print(len(x_train), len(x_test))

623 268


In [8]:
tree = DecisionTreeClassifier(max_depth=3, random_state=42)
tree.fit(x_train, y_train)
print("Tree train accuracy:", tree.score(x_train, y_train))
print("Tree test accuracy:", tree.score(x_test, y_test))

Tree train accuracy: 0.826645264847512
Tree test accuracy: 0.8059701492537313


In [9]:
knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(x_train, y_train)
print("kNN test accuracy:", knn.score(x_test, y_test))

kNN test accuracy: 0.6828358208955224


In [11]:
pred = tree.predict(x_test)
results = x_test.copy()
results["actual"] = y_test
results["predicted"] = pred
wrong = results[results["actual"] != results["predicted"]]
print("wrong:", len(wrong), "out of", len(results))
print(wrong.groupby(["Sex", "Pclass", "actual"]).size())

wrong: 52 out of 268
Sex  Pclass  actual
0    1       1         20
     2       1          2
     3       0          1
             1         11
1    2       0          5
     3       0         11
             1          2
dtype: int64


In [ ]:
#The model most often gets wrong men who survived, are mostly first class and third class, which it predicts died,
#and women in 2nd and 3rd class who died, which it predicts survived.